# Project 3 · Extended Solution — Inventory Service and Build Lifecycle

**Standalone advanced extension**: the same correctly validated inheritance
hierarchy, followed by an audit-enabled SKU catalog, atomic multi-component
build allocation and release, low-stock reporting, and JSON inventory export.

**Runtime:** modern Python 3.11+; **dependencies:** standard library only.
Run this notebook independently from the first notebook.

## 1 · Operational semantics

**Invariants:** `0 <= allocated <= total` per resource. `died(n)` retires
allocated damaged stock; `retire_available(n)` retires unused stock. Invalid
quantities or shortages reject entire requests. SKU is a **unique identifier
for a resource pool**, not an individual item serial number.

For catalog operations, perform mutations via the catalog to get audit
entries. Direct access to a returned `Resource` is intentionally possible
for exercise compatibility, but direct mutations bypass catalog auditing
and can invalidate outstanding build reservations. The catalog can detect
insufficient stock at release; it does not own exclusive Resource mutation
rights. A production system would enforce this with a dedicated persistence
boundary and an allocation ledger at item or reservation level.

JSON export saves **inventory quantities and hardware specifications only**:
events and build records are intentionally ephemeral. A loaded catalog
cannot release builds created before export. This is a deliberate teaching
boundary, not a complete production persistence solution.

## 2 · Foundation: validation, snapshots and classes

In [10]:
from __future__ import annotations

from dataclasses import dataclass
from threading import RLock
from typing import Any


def validate_integer(
    value: object,
    *,
    name: str,
    minimum: int | None = None,
    maximum: int | None = None,
    bounds_message: str | None = None,
) -> int:
    """Return an integer after strict type and inclusive bound checks.

    Booleans are rejected: ``bool`` is an ``int`` subclass but not a count.
    """
    if type(value) is not int:
        raise TypeError(f"{name} must be an integer (bool is not accepted)")
    if minimum is not None and value < minimum:
        raise ValueError(bounds_message or f"{name} must be >= {minimum}")
    if maximum is not None and value > maximum:
        raise ValueError(bounds_message or f"{name} must be <= {maximum}")
    return value


def validate_text(value: object, *, name: str) -> str:
    """Accept a non-blank string, stripping surrounding whitespace."""
    if not isinstance(value, str):
        raise TypeError(f"{name} must be a string")
    result = value.strip()
    if not result:
        raise ValueError(f"{name} must not be blank")
    return result


@dataclass(frozen=True, slots=True)
class InventorySnapshot:
    """An immutable, consistent view of the stock at one instant."""

    name: str
    category: str
    manufacturer: str
    total: int
    allocated: int
    available: int

In [11]:
class Resource:
    """A stock pool for one type of hardware, not an individual physical item.

    Invariant: 0 <= allocated <= total. Public configuration is read-only.
    ``died(n)`` retires n *allocated* damaged items; call ``freeup(n)``
    instead for working items returned to stock.
    """

    __slots__ = ("_name", "_manufacturer", "_total", "_allocated", "_lock")

    def __init__(
        self,
        name: str,
        manufacturer: str,
        *,
        total: int = 0,
        allocated: int = 0,
    ) -> None:
        name = validate_text(name, name="name")
        manufacturer = validate_text(manufacturer, name="manufacturer")
        total = validate_integer(total, name="total", minimum=0)
        allocated = validate_integer(allocated, name="allocated", minimum=0)
        if allocated > total:
            raise ValueError("allocated cannot exceed total")
        self._name = name
        self._manufacturer = manufacturer
        self._total = total
        self._allocated = allocated
        self._lock = RLock()

    @property
    def name(self) -> str:
        return self._name

    @property
    def manufacturer(self) -> str:
        return self._manufacturer

    @property
    def manufacturer_name(self) -> str:
        """Compatibility alias for exercise wording."""
        return self._manufacturer

    @property
    def total(self) -> int:
        with self._lock:
            return self._total

    @property
    def allocated(self) -> int:
        with self._lock:
            return self._allocated

    @property
    def available(self) -> int:
        with self._lock:
            return self._total - self._allocated

    @property
    def category(self) -> str:
        return type(self).__name__.lower()

    def snapshot(self) -> InventorySnapshot:
        with self._lock:
            return InventorySnapshot(
                name=self.name,
                category=self.category,
                manufacturer=self.manufacturer,
                total=self._total,
                allocated=self._allocated,
                available=self._total - self._allocated,
            )

    def _check_quantity(self, quantity: int) -> int:
        return validate_integer(quantity, name="quantity", minimum=1)

    def claim(self, n: int) -> None:
        """Allocate n available items; reject the entire request if short."""
        n = self._check_quantity(n)
        with self._lock:
            if n > self._total - self._allocated:
                raise ValueError("insufficient available inventory")
            self._allocated += n

    def freeup(self, n: int) -> None:
        """Return n allocated, working items to available stock."""
        n = self._check_quantity(n)
        with self._lock:
            if n > self._allocated:
                raise ValueError("cannot free more items than are allocated")
            self._allocated -= n

    def died(self, n: int) -> None:
        """Retire n damaged allocated items (total and allocated decrease)."""
        n = self._check_quantity(n)
        with self._lock:
            if n > self._allocated:
                raise ValueError("cannot retire more damaged items than allocated")
            self._allocated -= n
            self._total -= n


    def retire_available(self, n: int) -> None:
        """Retire n *unallocated* items without disturbing existing builds."""
        n = self._check_quantity(n)
        with self._lock:
            if n > self._total - self._allocated:
                raise ValueError("cannot retire more items than are available")
            self._total -= n

    def purchased(self, n: int) -> None:
        """Add n newly purchased items to available inventory."""
        n = self._check_quantity(n)
        with self._lock:
            self._total += n

    def _spec_repr(self) -> str:
        """Extension point for subclass-specific constructor arguments."""
        return ""

    def __repr__(self) -> str:
        with self._lock:
            extras = self._spec_repr()
            if extras:
                extras = ", " + extras
            return (
                f"{type(self).__name__}(name={self.name!r}, "
                f"manufacturer={self.manufacturer!r}{extras}, "
                f"total={self._total}, allocated={self._allocated})"
            )

    def __str__(self) -> str:
        return self.name

In [12]:
class CPU(Resource):
    """A CPU stock pool with immutable processor specifications."""

    __slots__ = ("_cores", "_socket", "_power_watts")

    def __init__(
        self,
        name: str,
        manufacturer: str,
        *,
        cores: int,
        socket: str,
        power_watts: int,
        total: int = 0,
        allocated: int = 0,
    ) -> None:
        cores = validate_integer(cores, name="cores", minimum=1)
        socket = validate_text(socket, name="socket")
        power_watts = validate_integer(power_watts, name="power_watts", minimum=1)
        super().__init__(name, manufacturer, total=total, allocated=allocated)
        self._cores = cores
        self._socket = socket
        self._power_watts = power_watts

    @property
    def cores(self) -> int:
        return self._cores

    @property
    def socket(self) -> str:
        return self._socket

    @property
    def power_watts(self) -> int:
        return self._power_watts

    def _spec_repr(self) -> str:
        return (
            f"cores={self.cores!r}, socket={self.socket!r}, "
            f"power_watts={self.power_watts!r}"
        )


class Storage(Resource):
    """Intermediate class shared by HDD and SSD."""

    __slots__ = ("_capacity_GB",)

    def __init__(
        self,
        name: str,
        manufacturer: str,
        *,
        capacity_GB: int,
        total: int = 0,
        allocated: int = 0,
    ) -> None:
        capacity_GB = validate_integer(capacity_GB, name="capacity_GB", minimum=1)
        super().__init__(name, manufacturer, total=total, allocated=allocated)
        self._capacity_GB = capacity_GB

    @property
    def capacity_GB(self) -> int:
        return self._capacity_GB

    def _spec_repr(self) -> str:
        return f"capacity_GB={self.capacity_GB!r}"


class HDD(Storage):
    """Mechanical disk inventory with form-factor and RPM specifications."""

    __slots__ = ("_size", "_rpm")

    def __init__(
        self,
        name: str,
        manufacturer: str,
        *,
        capacity_GB: int,
        size: str,
        rpm: int,
        total: int = 0,
        allocated: int = 0,
    ) -> None:
        size = validate_text(size, name="size")
        rpm = validate_integer(rpm, name="rpm", minimum=1)
        super().__init__(
            name, manufacturer, capacity_GB=capacity_GB,
            total=total, allocated=allocated,
        )
        self._size = size
        self._rpm = rpm

    @property
    def size(self) -> str:
        return self._size

    @property
    def rpm(self) -> int:
        return self._rpm

    def _spec_repr(self) -> str:
        return f"{super()._spec_repr()}, size={self.size!r}, rpm={self.rpm!r}"


class SSD(Storage):
    """Solid-state disk inventory with a read-only interface specification."""

    __slots__ = ("_interface",)

    def __init__(
        self,
        name: str,
        manufacturer: str,
        *,
        capacity_GB: int,
        interface: str,
        total: int = 0,
        allocated: int = 0,
    ) -> None:
        interface = validate_text(interface, name="interface")
        super().__init__(
            name, manufacturer, capacity_GB=capacity_GB,
            total=total, allocated=allocated,
        )
        self._interface = interface

    @property
    def interface(self) -> str:
        return self._interface

    def _spec_repr(self) -> str:
        return f"{super()._spec_repr()}, interface={self.interface!r}"

## 3 · Audit types and SKU catalog

In [13]:
from collections.abc import Mapping
from contextlib import ExitStack
from dataclasses import replace
from datetime import datetime, timezone
from itertools import count
from types import MappingProxyType
import json

In [14]:
@dataclass(frozen=True, slots=True)
class AuditEvent:
    """Immutable audit entry for an operation made via InventoryCatalog."""

    sequence: int
    at_utc: datetime
    sku: str
    action: str
    quantity: int
    before: InventorySnapshot
    after: InventorySnapshot


@dataclass(frozen=True, slots=True)
class BuildRecord:
    """Immutable description of a multi-component build allocation."""

    build_id: int
    items: tuple[tuple[str, int], ...]
    active: bool


class InventoryCatalog:
    """A SKU-indexed service with serialized stock mutations and audit entries.

    For complete audit coverage, mutate stock *through the catalog* only.
    Catalog locking protects its own operations; it cannot intercept direct
    calls made on a Resource elsewhere. JSON exports stock only, not the
    in-memory audit history or build records.
    """

    __slots__ = ("_items", "_events", "_builds", "_next_event", "_next_build", "_lock")
    _RESOURCE_TYPES = MappingProxyType({
        "Resource": Resource, "CPU": CPU, "Storage": Storage,
        "HDD": HDD, "SSD": SSD,
    })

    def __init__(self) -> None:
        self._items: dict[str, Resource] = {}
        self._events: list[AuditEvent] = []
        self._builds: dict[int, BuildRecord] = {}
        self._next_event = count(1)
        self._next_build = count(1)
        self._lock = RLock()

    def add(self, sku: str, resource: Resource) -> None:
        """Register a resource with a unique SKU and unique object identity."""
        sku = validate_text(sku, name="sku")
        if not isinstance(resource, Resource):
            raise TypeError("resource must be an instance of Resource")
        with self._lock:
            if sku in self._items:
                raise ValueError(f"SKU already registered: {sku}")
            if any(existing is resource for existing in self._items.values()):
                raise ValueError("the same Resource object cannot be registered twice")
            self._items[sku] = resource

    def get(self, sku: str) -> Resource:
        sku = validate_text(sku, name="sku")
        with self._lock:
            return self._items[sku]

    def list_inventory(self) -> tuple[tuple[str, InventorySnapshot], ...]:
        """Return sorted SKU/snapshot pairs without exposing internal mappings."""
        with self._lock:
            return tuple((sku, self._items[sku].snapshot()) for sku in sorted(self._items))

    def low_stock(self, threshold: int = 0) -> tuple[tuple[str, InventorySnapshot], ...]:
        threshold = validate_integer(threshold, name="threshold", minimum=0)
        return tuple(
            (sku, snapshot) for sku, snapshot in self.list_inventory()
            if snapshot.available <= threshold
        )

    @property
    def audit_log(self) -> tuple[AuditEvent, ...]:
        with self._lock:
            return tuple(self._events)

    def _record(
        self,
        *,
        sku: str,
        action: str,
        quantity: int,
        before: InventorySnapshot,
        after: InventorySnapshot,
    ) -> None:
        self._events.append(AuditEvent(
            sequence=next(self._next_event),
            at_utc=datetime.now(timezone.utc),
            sku=sku,
            action=action,
            quantity=quantity,
            before=before,
            after=after,
        ))

    def _operate(self, sku: str, n: int, method: str) -> InventorySnapshot:
        """Perform a single stock mutation and append its audit entry."""
        sku = validate_text(sku, name="sku")
        n = validate_integer(n, name="quantity", minimum=1)
        with self._lock:
            item = self._items[sku]
            with item._lock:
                before = item.snapshot()
                getattr(item, method)(n)
                after = item.snapshot()
                self._record(
                    sku=sku, action=method, quantity=n, before=before, after=after,
                )
                return after

    def claim(self, sku: str, n: int) -> InventorySnapshot:
        return self._operate(sku, n, "claim")

    def freeup(self, sku: str, n: int) -> InventorySnapshot:
        return self._operate(sku, n, "freeup")

    def died(self, sku: str, n: int) -> InventorySnapshot:
        return self._operate(sku, n, "died")

    def purchased(self, sku: str, n: int) -> InventorySnapshot:
        return self._operate(sku, n, "purchased")

    def retire_available(self, sku: str, n: int) -> InventorySnapshot:
        return self._operate(sku, n, "retire_available")

    def _normalize_request(self, requests: Mapping[str, int]) -> tuple[tuple[str, int], ...]:
        if not isinstance(requests, Mapping):
            raise TypeError("requests must be a mapping of SKUs to positive quantities")
        if not requests:
            raise ValueError("a build must contain at least one resource")
        result = []
        normalized_skus = set()
        for sku, quantity in requests.items():
            normalized = validate_text(sku, name="sku")
            if normalized in normalized_skus:
                raise ValueError("duplicate SKU after whitespace normalization")
            normalized_skus.add(normalized)
            result.append((normalized, validate_integer(quantity, name="quantity", minimum=1)))
        return tuple(sorted(result))

    def allocate_build(self, requests: Mapping[str, int]) -> BuildRecord:
        """All-or-nothing allocation across several SKUs.

        Acquire resource locks in sorted SKU order. Validate *every* item before
        making changes; restore original counts and truncate events on an
        unexpected commit error. Individual catalog mutations are serialized
        by the catalog lock, and resource locks exclude concurrent direct
        resource mutations during this transaction.
        """
        items = self._normalize_request(requests)
        with self._lock:
            resources = [(sku, count_, self._items[sku]) for sku, count_ in items]
            with ExitStack() as stack:
                for _, _, resource in resources:
                    stack.enter_context(resource._lock)
                before = {sku: resource.snapshot() for sku, _, resource in resources}
                for sku, quantity, resource in resources:
                    if resource.available < quantity:
                        raise ValueError(f"insufficient available inventory for {sku}")
                event_length = len(self._events)
                try:
                    for sku, quantity, resource in resources:
                        resource.claim(quantity)
                        self._record(
                            sku=sku, action="build_claim", quantity=quantity,
                            before=before[sku], after=resource.snapshot(),
                        )
                    record = BuildRecord(next(self._next_build), items, active=True)
                    self._builds[record.build_id] = record
                    return record
                except Exception:
                    for sku, _, resource in resources:
                        resource._total = before[sku].total
                        resource._allocated = before[sku].allocated
                    del self._events[event_length:]
                    raise

    def get_build(self, build_id: int) -> BuildRecord:
        build_id = validate_integer(build_id, name="build_id", minimum=1)
        with self._lock:
            return self._builds[build_id]

    def release_build(self, build_id: int) -> BuildRecord:
        """Return all components from an active build, only once."""
        build_id = validate_integer(build_id, name="build_id", minimum=1)
        with self._lock:
            record = self._builds[build_id]
            if not record.active:
                raise ValueError("build has already been released")
            resources = [(sku, n, self._items[sku]) for sku, n in record.items]
            with ExitStack() as stack:
                for _, _, resource in resources:
                    stack.enter_context(resource._lock)
                before = {sku: resource.snapshot() for sku, _, resource in resources}
                for sku, quantity, resource in resources:
                    if resource.allocated < quantity:
                        raise ValueError(f"insufficient allocated inventory for {sku}")
                event_length = len(self._events)
                try:
                    for sku, quantity, resource in resources:
                        resource.freeup(quantity)
                        self._record(
                            sku=sku, action="build_release", quantity=quantity,
                            before=before[sku], after=resource.snapshot(),
                        )
                    released = replace(record, active=False)
                    self._builds[build_id] = released
                    return released
                except Exception:
                    for sku, _, resource in resources:
                        resource._total = before[sku].total
                        resource._allocated = before[sku].allocated
                    del self._events[event_length:]
                    raise

    def to_json(self, *, indent: int | None = 2) -> str:
        """Serialize inventory stock, not audits or build records, to JSON."""
        with self._lock:
            payload = []
            for sku, item in sorted(self._items.items()):
                with item._lock:
                    data = {
                        "sku": sku, "kind": type(item).__name__, "name": item.name,
                        "manufacturer": item.manufacturer,
                        "total": item.total, "allocated": item.allocated,
                    }
                    if isinstance(item, CPU):
                        data.update(cores=item.cores, socket=item.socket,
                                    power_watts=item.power_watts)
                    if isinstance(item, Storage):
                        data["capacity_GB"] = item.capacity_GB
                    if isinstance(item, HDD):
                        data.update(size=item.size, rpm=item.rpm)
                    if isinstance(item, SSD):
                        data["interface"] = item.interface
                    payload.append(data)
            return json.dumps({"schema_version": 1, "resources": payload}, indent=indent)

    @classmethod
    def from_json(cls, text: str) -> InventoryCatalog:
        """Reconstruct trusted schema fields with an explicit type allowlist."""
        data = json.loads(text)
        if not isinstance(data, dict) or data.get("schema_version") != 1:
            raise ValueError("unsupported inventory JSON schema")
        resources = data.get("resources")
        if not isinstance(resources, list):
            raise ValueError("resources must be a list")
        catalog = cls()
        for entry in resources:
            if not isinstance(entry, dict):
                raise ValueError("each resource must be an object")
            fields = dict(entry)
            sku = fields.pop("sku")
            kind = fields.pop("kind")
            if not isinstance(kind, str) or kind not in cls._RESOURCE_TYPES:
                raise ValueError(f"unknown resource type: {kind!r}")
            resource = cls._RESOURCE_TYPES[kind](**fields)
            catalog.add(sku, resource)
        return catalog

## 4 · Usage: build allocation, audit trail, stock reports, JSON

In [15]:
catalog = InventoryCatalog()
catalog.add("CPU-2700", CPU(
    "Ryzen 7 2700", "AMD", cores=8, socket="AM4", power_watts=65, total=5,
))
catalog.add("SSD-970", SSD(
    "970 EVO", "Samsung", capacity_GB=1000, interface="NVMe", total=3,
))
catalog.add("HDD-BAR", HDD(
    "BarraCuda", "Seagate", capacity_GB=2000, size='3.5"', rpm=7200, total=2,
))

build = catalog.allocate_build({"CPU-2700": 1, "SSD-970": 1})
assert build.active and build.items == (("CPU-2700", 1), ("SSD-970", 1))
assert catalog.get("CPU-2700").allocated == 1
assert catalog.get("SSD-970").allocated == 1
released = catalog.release_build(build.build_id)
assert not released.active
assert catalog.get("CPU-2700").allocated == 0
catalog.retire_available("HDD-BAR", 1)
assert catalog.get("HDD-BAR").total == 1
print("Inventory:", catalog.list_inventory())
print("Audit entries:", len(catalog.audit_log))
print("Low stock (<= 1):", catalog.low_stock(1))
print("JSON inventory excerpt:", catalog.to_json()[:230], "...")

Inventory: (('CPU-2700', InventorySnapshot(name='Ryzen 7 2700', category='cpu', manufacturer='AMD', total=5, allocated=0, available=5)), ('HDD-BAR', InventorySnapshot(name='BarraCuda', category='hdd', manufacturer='Seagate', total=1, allocated=0, available=1)), ('SSD-970', InventorySnapshot(name='970 EVO', category='ssd', manufacturer='Samsung', total=3, allocated=0, available=3)))
Audit entries: 5
Low stock (<= 1): (('HDD-BAR', InventorySnapshot(name='BarraCuda', category='hdd', manufacturer='Seagate', total=1, allocated=0, available=1)),)
JSON inventory excerpt: {
  "schema_version": 1,
  "resources": [
    {
      "sku": "CPU-2700",
      "kind": "CPU",
      "name": "Ryzen 7 2700",
      "manufacturer": "AMD",
      "total": 5,
      "allocated": 0,
      "cores": 8,
      "socket": "AM ...


## 5 · Automated regression and extension tests

The entire advanced test suite is repeated in this notebook to make it
self-contained. Further tests cover SKU uniqueness, atomic multi-SKU
allocation, one-time release, audit chronology, stock retirement,
multithreaded requests, JSON round trips and corrupted input.

In [16]:
import unittest
from concurrent.futures import ThreadPoolExecutor


class ValidatorTests(unittest.TestCase):
    def test_integer_limits_inclusive(self):
        self.assertEqual(validate_integer(2, name="x", minimum=2, maximum=4), 2)
        self.assertEqual(validate_integer(4, name="x", minimum=2, maximum=4), 4)

    def test_integer_rejects_float_string_bool(self):
        for value in (1.0, "1", True, False, None):
            with self.subTest(value=value), self.assertRaises(TypeError):
                validate_integer(value, name="x")

    def test_integer_minimum_and_custom_message(self):
        with self.assertRaisesRegex(ValueError, "custom"):
            validate_integer(0, name="x", minimum=1, bounds_message="custom")

    def test_integer_maximum(self):
        with self.assertRaises(ValueError):
            validate_integer(5, name="x", maximum=4)

    def test_text_normalizes_whitespace(self):
        self.assertEqual(validate_text("  AMD  ", name="x"), "AMD")

    def test_text_rejects_bad_inputs(self):
        for value, exception in ((5, TypeError), ("  ", ValueError)):
            with self.subTest(value=value), self.assertRaises(exception):
                validate_text(value, name="x")


class ResourceTests(unittest.TestCase):
    def setUp(self):
        self.resource = Resource("  Fan  ", "  Noctua  ", total=5, allocated=2)

    def test_initial_state_and_names(self):
        r = self.resource
        self.assertEqual((r.name, r.manufacturer), ("Fan", "Noctua"))
        self.assertEqual(r.manufacturer_name, "Noctua")
        self.assertEqual((r.total, r.allocated, r.available), (5, 2, 3))
        self.assertEqual(r.category, "resource")

    def test_default_counts(self):
        r = Resource("Fan", "Noctua")
        self.assertEqual((r.total, r.allocated, r.available), (0, 0, 0))

    def test_invalid_initial_counts(self):
        for kwargs in ({"total": -1}, {"allocated": -1}, {"total": 1, "allocated": 2}):
            with self.subTest(kwargs=kwargs), self.assertRaises(ValueError):
                Resource("Fan", "Noctua", **kwargs)

    def test_invalid_count_types(self):
        for kwargs in ({"total": True}, {"allocated": 1.5}):
            with self.subTest(kwargs=kwargs), self.assertRaises(TypeError):
                Resource("Fan", "Noctua", **kwargs)

    def test_reject_empty_names(self):
        with self.assertRaises(ValueError):
            Resource(" ", "Company")
        with self.assertRaises(ValueError):
            Resource("Item", " ")

    def test_properties_read_only(self):
        for attribute, value in (
            ("name", "Other"), ("manufacturer", "Other"),
            ("manufacturer_name", "Other"), ("total", 100),
            ("allocated", 100), ("available", 100), ("category", "other"),
        ):
            with self.subTest(attribute=attribute), self.assertRaises(AttributeError):
                setattr(self.resource, attribute, value)

    def test_claim_success(self):
        self.resource.claim(3)
        self.assertEqual((self.resource.allocated, self.resource.available), (5, 0))

    def test_claim_failure_is_atomic(self):
        before = self.resource.snapshot()
        with self.assertRaises(ValueError):
            self.resource.claim(4)
        self.assertEqual(self.resource.snapshot(), before)

    def test_freeup_success(self):
        self.resource.freeup(2)
        self.assertEqual((self.resource.allocated, self.resource.available), (0, 5))

    def test_freeup_failure_is_atomic(self):
        before = self.resource.snapshot()
        with self.assertRaises(ValueError):
            self.resource.freeup(3)
        self.assertEqual(self.resource.snapshot(), before)

    def test_died_retires_allocated(self):
        self.resource.died(2)
        self.assertEqual((self.resource.total, self.resource.allocated, self.resource.available), (3, 0, 3))

    def test_died_failure_is_atomic(self):
        before = self.resource.snapshot()
        with self.assertRaises(ValueError):
            self.resource.died(3)
        self.assertEqual(self.resource.snapshot(), before)

    def test_purchased(self):
        self.resource.purchased(2)
        self.assertEqual((self.resource.total, self.resource.allocated, self.resource.available), (7, 2, 5))

    def test_all_methods_reject_nonpositive_quantity(self):
        for method in ("claim", "freeup", "died", "purchased"):
            for quantity in (0, -1):
                with self.subTest(method=method, quantity=quantity), self.assertRaises(ValueError):
                    getattr(self.resource, method)(quantity)

    def test_all_methods_reject_bool_and_float(self):
        for method in ("claim", "freeup", "died", "purchased"):
            for quantity in (True, 1.2):
                with self.subTest(method=method, quantity=quantity), self.assertRaises(TypeError):
                    getattr(self.resource, method)(quantity)

    def test_snapshot_is_immutable(self):
        with self.assertRaises(AttributeError):
            self.resource.snapshot().total = 100

    def test_str_repr_round_trip(self):
        self.assertEqual(str(self.resource), "Fan")
        reconstructed = eval(repr(self.resource), {"Resource": Resource})
        self.assertEqual(reconstructed.snapshot(), self.resource.snapshot())

    def test_concurrent_claims_do_not_overallocate(self):
        resource = Resource("Fan", "Noctua", total=30)
        def try_claim(_):
            try:
                resource.claim(1)
                return True
            except ValueError:
                return False
        with ThreadPoolExecutor(max_workers=12) as pool:
            outcomes = list(pool.map(try_claim, range(100)))
        self.assertEqual(sum(outcomes), 30)
        self.assertEqual((resource.total, resource.allocated, resource.available), (30, 30, 0))


class SubclassTests(unittest.TestCase):
    def test_cpu_constructor_and_specs(self):
        cpu = CPU("Processor", "AMD", cores=8, socket="AM4", power_watts=65, total=5)
        self.assertIsInstance(cpu, Resource)
        self.assertEqual((cpu.cores, cpu.socket, cpu.power_watts), (8, "AM4", 65))
        self.assertEqual(cpu.category, "cpu")

    def test_cpu_specs_are_read_only(self):
        cpu = CPU("Processor", "AMD", cores=8, socket="AM4", power_watts=65)
        for key in ("cores", "socket", "power_watts"):
            with self.subTest(key=key), self.assertRaises(AttributeError):
                setattr(cpu, key, 99)

    def test_cpu_rejects_bad_specs(self):
        for overrides, error in (
            ({"cores": 0}, ValueError), ({"cores": True}, TypeError),
            ({"socket": "  "}, ValueError), ({"power_watts": -5}, ValueError),
        ):
            kwargs = {"cores": 8, "socket": "AM4", "power_watts": 65} | overrides
            with self.subTest(overrides=overrides), self.assertRaises(error):
                CPU("Processor", "AMD", **kwargs)

    def test_storage_inheritance_and_capacity(self):
        disk = Storage("Disk", "WD", capacity_GB=500)
        self.assertIsInstance(disk, Resource)
        self.assertEqual(disk.capacity_GB, 500)
        with self.assertRaises(AttributeError):
            disk.capacity_GB = 1000
        with self.assertRaises(ValueError):
            Storage("Disk", "WD", capacity_GB=0)

    def test_hdd_inherits_storage(self):
        disk = HDD("Disk", "WD", capacity_GB=2000, size='3.5"', rpm=7200)
        self.assertIsInstance(disk, Storage)
        self.assertEqual((disk.size, disk.rpm, disk.capacity_GB), ('3.5"', 7200, 2000))
        self.assertEqual(disk.category, "hdd")

    def test_hdd_bad_specs(self):
        with self.assertRaises(TypeError):
            HDD("Disk", "WD", capacity_GB=500, size='3.5"', rpm=True)
        with self.assertRaises(ValueError):
            HDD("Disk", "WD", capacity_GB=500, size=" ", rpm=7200)
        with self.assertRaises(ValueError):
            HDD("Disk", "WD", capacity_GB=500, size='3.5"', rpm=0)

    def test_ssd_inherits_storage(self):
        disk = SSD("Disk", "Samsung", capacity_GB=1000, interface="NVMe", total=2)
        self.assertIsInstance(disk, Storage)
        self.assertEqual((disk.interface, disk.capacity_GB, disk.category), ("NVMe", 1000, "ssd"))
        with self.assertRaises(AttributeError):
            disk.interface = "SATA"
        with self.assertRaises(ValueError):
            SSD("Disk", "Samsung", capacity_GB=1000, interface=" ")

    def test_subclass_repr_round_trip(self):
        items = [
            CPU("Processor", "AMD", cores=8, socket="AM4", power_watts=65, total=3, allocated=2),
            Storage("Disk", "WD", capacity_GB=500, total=2),
            HDD("Disk", "WD", capacity_GB=2000, size='3.5"', rpm=7200, total=2),
            SSD("Disk", "Samsung", capacity_GB=1000, interface="NVMe", total=2),
        ]
        for item in items:
            with self.subTest(cls=type(item).__name__):
                reconstructed = eval(repr(item), {
                    "CPU": CPU, "Storage": Storage, "HDD": HDD, "SSD": SSD,
                })
                self.assertEqual(reconstructed.snapshot(), item.snapshot())
                self.assertEqual(repr(reconstructed), repr(item))

    def test_subclass_inherits_inventory_methods(self):
        cpu = CPU("Processor", "AMD", cores=8, socket="AM4", power_watts=65, total=4)
        cpu.claim(2)
        cpu.died(1)
        cpu.freeup(1)
        cpu.purchased(2)
        self.assertEqual((cpu.total, cpu.allocated, cpu.available), (5, 0, 5))

In [17]:
class ExtendedResourceTests(unittest.TestCase):
    def test_retire_available_changes_total_not_allocated(self):
        r = Resource("CPU", "AMD", total=5, allocated=2)
        r.retire_available(2)
        self.assertEqual((r.total, r.allocated, r.available), (3, 2, 1))

    def test_retire_available_rejects_excess_without_changes(self):
        r = Resource("CPU", "AMD", total=5, allocated=4)
        before = r.snapshot()
        with self.assertRaises(ValueError):
            r.retire_available(2)
        self.assertEqual(r.snapshot(), before)

    def test_retire_available_validates_quantity(self):
        r = Resource("CPU", "AMD", total=5)
        for value, expected in ((0, ValueError), (True, TypeError)):
            with self.subTest(value=value), self.assertRaises(expected):
                r.retire_available(value)


def make_catalog() -> InventoryCatalog:
    c = InventoryCatalog()
    c.add("CPU", CPU("Ryzen", "AMD", cores=8, socket="AM4", power_watts=65, total=4))
    c.add("SSD", SSD("970 EVO", "Samsung", capacity_GB=1000, interface="NVMe", total=2))
    c.add("HDD", HDD("BarraCuda", "Seagate", capacity_GB=2000, size='3.5"', rpm=7200, total=3))
    return c


class CatalogTests(unittest.TestCase):
    def setUp(self):
        self.catalog = make_catalog()

    def test_get_and_sorted_list(self):
        self.assertIsInstance(self.catalog.get("CPU"), CPU)
        self.assertEqual([sku for sku, _ in self.catalog.list_inventory()], ["CPU", "HDD", "SSD"])

    def test_add_rejects_duplicate_sku(self):
        with self.assertRaises(ValueError):
            self.catalog.add("CPU", Resource("other", "maker"))

    def test_add_rejects_same_object_under_different_sku(self):
        with self.assertRaises(ValueError):
            self.catalog.add("CPU2", self.catalog.get("CPU"))

    def test_add_rejects_bad_type_and_sku(self):
        with self.assertRaises(TypeError):
            self.catalog.add("GPU", object())
        with self.assertRaises(ValueError):
            self.catalog.add("   ", Resource("x", "y"))

    def test_get_missing_raises_keyerror(self):
        with self.assertRaises(KeyError):
            self.catalog.get("MISSING")

    def test_catalog_single_mutations_audited(self):
        self.catalog.claim("CPU", 2)
        self.catalog.freeup("CPU", 1)
        self.catalog.died("CPU", 1)
        self.catalog.purchased("CPU", 3)
        self.catalog.retire_available("CPU", 2)
        self.assertEqual((self.catalog.get("CPU").total, self.catalog.get("CPU").allocated), (4, 0))
        events = self.catalog.audit_log
        self.assertEqual([e.action for e in events], [
            "claim", "freeup", "died", "purchased", "retire_available",
        ])
        self.assertEqual([e.sequence for e in events], [1, 2, 3, 4, 5])
        self.assertTrue(all(event.at_utc.utcoffset().total_seconds() == 0 for event in events))
        self.assertEqual(events[0].before.allocated, 0)
        self.assertEqual(events[0].after.allocated, 2)

    def test_failed_catalog_operation_has_no_audit(self):
        before = self.catalog.list_inventory()
        with self.assertRaises(ValueError):
            self.catalog.claim("SSD", 3)
        self.assertEqual(self.catalog.list_inventory(), before)
        self.assertFalse(self.catalog.audit_log)

    def test_catalog_validates_quantity(self):
        with self.assertRaises(TypeError):
            self.catalog.claim("CPU", True)
        with self.assertRaises(ValueError):
            self.catalog.purchased("CPU", 0)

    def test_audit_log_immutable_view(self):
        self.catalog.claim("CPU", 1)
        with self.assertRaises(AttributeError):
            self.catalog.audit_log[0].quantity = 100
        self.assertIsInstance(self.catalog.audit_log, tuple)

    def test_low_stock_inclusive_and_sorted(self):
        self.catalog.claim("SSD", 2)
        self.catalog.claim("CPU", 3)
        self.assertEqual([sku for sku, _ in self.catalog.low_stock(1)], ["CPU", "SSD"])
        with self.assertRaises(ValueError):
            self.catalog.low_stock(-1)
        with self.assertRaises(TypeError):
            self.catalog.low_stock(True)

    def test_build_allocates_all_items(self):
        record = self.catalog.allocate_build({"SSD": 1, "CPU": 2})
        self.assertTrue(record.active)
        self.assertEqual(record.items, (("CPU", 2), ("SSD", 1)))
        self.assertEqual(self.catalog.get("CPU").allocated, 2)
        self.assertEqual(self.catalog.get("SSD").allocated, 1)
        self.assertEqual([e.action for e in self.catalog.audit_log], ["build_claim", "build_claim"])
        self.assertEqual(self.catalog.get_build(record.build_id), record)

    def test_build_failure_is_all_or_nothing(self):
        before = self.catalog.list_inventory()
        with self.assertRaisesRegex(ValueError, "SSD"):
            self.catalog.allocate_build({"CPU": 2, "SSD": 3})
        self.assertEqual(self.catalog.list_inventory(), before)
        self.assertFalse(self.catalog.audit_log)

    def test_build_with_unknown_sku_is_all_or_nothing(self):
        before = self.catalog.list_inventory()
        with self.assertRaises(KeyError):
            self.catalog.allocate_build({"CPU": 1, "MISSING": 1})
        self.assertEqual(self.catalog.list_inventory(), before)
        self.assertFalse(self.catalog.audit_log)

    def test_build_rejects_empty_or_wrong_request(self):
        with self.assertRaises(ValueError):
            self.catalog.allocate_build({})
        with self.assertRaises(TypeError):
            self.catalog.allocate_build([("CPU", 1)])

    def test_build_rejects_bad_quantity(self):
        for quantity, error in ((0, ValueError), (True, TypeError), (1.5, TypeError)):
            with self.subTest(quantity=quantity), self.assertRaises(error):
                self.catalog.allocate_build({"CPU": quantity})

    def test_build_rejects_normalized_duplicate_sku(self):
        with self.assertRaises(ValueError):
            self.catalog.allocate_build({"CPU": 1, " CPU ": 1})

    def test_release_build_is_one_time_and_audited(self):
        record = self.catalog.allocate_build({"CPU": 1, "SSD": 1})
        released = self.catalog.release_build(record.build_id)
        self.assertFalse(released.active)
        self.assertEqual(self.catalog.get("CPU").allocated, 0)
        self.assertEqual(self.catalog.get("SSD").allocated, 0)
        self.assertEqual([e.action for e in self.catalog.audit_log], [
            "build_claim", "build_claim", "build_release", "build_release",
        ])
        with self.assertRaises(ValueError):
            self.catalog.release_build(record.build_id)

    def test_release_preserves_all_on_insufficient_allocated(self):
        record = self.catalog.allocate_build({"CPU": 1, "SSD": 1})
        self.catalog.freeup("SSD", 1)  # external interference with build reservations
        before = self.catalog.list_inventory()
        event_count = len(self.catalog.audit_log)
        with self.assertRaisesRegex(ValueError, "SSD"):
            self.catalog.release_build(record.build_id)
        self.assertEqual(self.catalog.list_inventory(), before)
        self.assertEqual(len(self.catalog.audit_log), event_count)
        self.assertTrue(self.catalog.get_build(record.build_id).active)

    def test_build_records_are_immutable(self):
        record = self.catalog.allocate_build({"CPU": 1})
        with self.assertRaises(AttributeError):
            record.active = False

    def test_catalog_concurrent_claims(self):
        def work(_):
            try:
                self.catalog.claim("CPU", 1)
                return True
            except ValueError:
                return False
        with ThreadPoolExecutor(max_workers=16) as pool:
            results = list(pool.map(work, range(50)))
        self.assertEqual(sum(results), 4)
        self.assertEqual(self.catalog.get("CPU").allocated, 4)
        self.assertEqual(len(self.catalog.audit_log), 4)

    def test_json_round_trip_all_resource_types(self):
        self.catalog.add("GEN", Resource("Generic", "Maker", total=3, allocated=1))
        self.catalog.add("STOR", Storage("Generic disk", "Maker", capacity_GB=500, total=2))
        self.catalog.claim("CPU", 2)
        text = self.catalog.to_json(indent=None)
        loaded = InventoryCatalog.from_json(text)
        self.assertEqual(loaded.list_inventory(), self.catalog.list_inventory())
        for sku in ("CPU", "SSD", "HDD", "GEN", "STOR"):
            self.assertEqual(repr(loaded.get(sku)), repr(self.catalog.get(sku)))
        self.assertFalse(loaded.audit_log)

    def test_json_rejects_unknown_schema(self):
        with self.assertRaises(ValueError):
            InventoryCatalog.from_json('{"schema_version": 2, "resources": []}')

    def test_json_rejects_unknown_resource_kind(self):
        with self.assertRaises(ValueError):
            InventoryCatalog.from_json('{"schema_version": 1, "resources": [{"kind": "GPU", "sku": "x"}]}')

    def test_json_rejects_invalid_state(self):
        malformed = json.dumps({"schema_version": 1, "resources": [{
            "sku": "x", "kind": "Resource", "name": "X", "manufacturer": "Y",
            "total": 1, "allocated": 2,
        }]})
        with self.assertRaises(ValueError):
            InventoryCatalog.from_json(malformed)

    def test_json_rejects_non_list_resources(self):
        with self.assertRaises(ValueError):
            InventoryCatalog.from_json('{"schema_version": 1, "resources": {}}')

In [18]:
suite = unittest.TestSuite(
    unittest.defaultTestLoader.loadTestsFromTestCase(cls)
    for cls in (ValidatorTests, ResourceTests, SubclassTests, ExtendedResourceTests, CatalogTests)
)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "One or more Project 3 tests failed"
print(f"PASS: {result.testsRun} automated tests")

test_integer_limits_inclusive (__main__.ValidatorTests.test_integer_limits_inclusive) ... ok
test_integer_maximum (__main__.ValidatorTests.test_integer_maximum) ... ok
test_integer_minimum_and_custom_message (__main__.ValidatorTests.test_integer_minimum_and_custom_message) ... ok
test_integer_rejects_float_string_bool (__main__.ValidatorTests.test_integer_rejects_float_string_bool) ... ok
test_text_normalizes_whitespace (__main__.ValidatorTests.test_text_normalizes_whitespace) ... ok
test_text_rejects_bad_inputs (__main__.ValidatorTests.test_text_rejects_bad_inputs) ... ok
test_all_methods_reject_bool_and_float (__main__.ResourceTests.test_all_methods_reject_bool_and_float) ... ok
test_all_methods_reject_nonpositive_quantity (__main__.ResourceTests.test_all_methods_reject_nonpositive_quantity) ... ok
test_claim_failure_is_atomic (__main__.ResourceTests.test_claim_failure_is_atomic) ... ok
test_claim_success (__main__.ResourceTests.test_claim_success) ... ok
test_concurrent_claims_do_no

PASS: 61 automated tests


## 6 · Production evolution

- Store SKUs, stock movements, reservations and build state in transactional
  database tables; use transaction isolation or optimistic concurrency.
- Protect reservation quantities from ordinary `freeup`/`died` calls and
  allocate physical serial-numbered items if traceability matters.
- Persist audit events and build records together with stock in **one**
  transaction; reconcile totals periodically and add authorization.
- Consider a `Decimal` price, supplier, restock threshold, and cost reports
  only after defining accounting and business rules.
- Serialization uses a known class allowlist and JSON, never untrusted
  `eval` or pickle. For hostile input, add schema size and depth limits.